# 02 - Treino LoRA

Prepara o dataset no formato que o script pede e treina 2 configs pra comparar.

## 1. Clonar o repo

In [9]:
!git clone https://github.com/lramosc1512/atelie-generativo-LeonidIA.git
%cd atelie-generativo-LeonidIA
!ls dados/imagens | wc -l
!head -3 dados/legendas.txt

fatal: destination path 'atelie-generativo-LeonidIA' already exists and is not an empty directory.
/content/diffusers/examples/text_to_image/atelie-generativo-LeonidIA
50
img_001.jpg,estilo_brasilia, fachada do Palácio do Planalto de dia, colunas brancas curvas sustentando a marquise, fachada de vidro ao fundo, céu azul e jardim florido em primeiro plano
img_002.jpg,estilo_brasilia, Palácio do Planalto à noite, colunas brancas curvas iluminadas contra o céu escuro, marquise em destaque
img_003.jpg,estilo_brasilia, Congresso Nacional de dia, duas torres gêmeas ao centro, cúpula da Câmara dos Deputados em formato de tigela à direita, cúpula do Senado invertida e côncava à esquerda, céu azul com nuvens


## 2. Instalar dependencias e clonar o script oficial

In [10]:
!pip -q install transformers accelerate peft datasets
%cd /content
!git clone https://github.com/huggingface/diffusers
%cd diffusers
!pip -q install -e .
%cd examples/text_to_image
!pip -q install -r requirements.txt
!pip -q install -U torchao

/content
fatal: destination path 'diffusers' already exists and is not an empty directory.
/content/diffusers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for diffusers (pyproject.toml) ... done
/content/diffusers/examples/text_to_image
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.0 MB/s eta 0:00:00


## 3. Login HF

In [11]:
from huggingface_hub import login
login()

## 4. Converter dataset pro formato imagefolder

O script pede metadata.jsonl (file_name + text), não lê fontes.csv/legendas.txt direto.

In [12]:
import json
import shutil
from pathlib import Path

ORIGEM_IMAGENS = Path("/content/atelie-generativo-LeonidIA/dados/imagens")
LEGENDAS_TXT = Path("/content/atelie-generativo-LeonidIA/dados/legendas.txt")
PASTA_DATASET = Path("/content/dataset_estilo_brasilia")

PASTA_DATASET.mkdir(parents=True, exist_ok=True)

linhas_metadata = []
with open(LEGENDAS_TXT, "r", encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha:
            continue
        arquivo, legenda = linha.split(",", 1)
        origem = ORIGEM_IMAGENS / arquivo
        destino = PASTA_DATASET / arquivo
        shutil.copy(origem, destino)
        linhas_metadata.append({"file_name": arquivo, "text": legenda.strip()})

with open(PASTA_DATASET / "metadata.jsonl", "w", encoding="utf-8") as f:
    for item in linhas_metadata:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"{len(linhas_metadata)} imagens copiadas e metadata.jsonl gerado em {PASTA_DATASET}")

50 imagens copiadas e metadata.jsonl gerado em /content/dataset_estilo_brasilia


## 4.5 Montar o Drive

Assim os checkpoints salvam direto lá, não em /content (que some se a sessão cair).

In [13]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/atelie_generativo", exist_ok=True)
print("Drive montado. Checkpoints serao salvos em /content/drive/MyDrive/atelie_generativo/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montado. Checkpoints serao salvos em /content/drive/MyDrive/atelie_generativo/


## 5. Config 1 - rank=4, lr=1e-4, 1000 steps

Rank baixo + lr mais alto: treina mais rápido, menos risco de overfit com dataset pequeno.

In [14]:
%cd /content/diffusers/examples/text_to_image

!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset_estilo_brasilia" \
  --resolution=512 --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-4 --lr_scheduler="cosine" \
  --rank=4 \
  --mixed_precision="fp16" \
  --checkpointing_steps=250 \
  --validation_prompt="estilo_brasilia, fachada curva de concreto branco refletida em espelho d'agua ao entardecer" \
  --output_dir="/content/drive/MyDrive/atelie_generativo/lora_config1_rank4" \
  --push_to_hub --hub_model_id="lramosc1512/lora-estilo-brasilia-config1"

/content/diffusers/examples/text_to_image
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers

## 6. Config 2 - rank=8, lr=5e-5, 1500 steps

Rank maior captura mais detalhe, compensado com lr mais conservador. Rodar só depois da config 1 terminar.

In [15]:
%cd /content/diffusers/examples/text_to_image

!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset_estilo_brasilia" \
  --resolution=512 --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=5e-5 --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=250 \
  --validation_prompt="estilo_brasilia, fachada curva de concreto branco refletida em espelho d'agua ao entardecer" \
  --output_dir="/content/drive/MyDrive/atelie_generativo/lora_config2_rank8" \
  --push_to_hub --hub_model_id="lramosc1512/lora-estilo-brasilia-config2"

/content/diffusers/examples/text_to_image
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers

## 7. Imagens de validação

Geradas a cada 250 passos com o validation_prompt - dá pra ver a evolução do estilo nos logs acima. Vai servir pra Etapa 3.

## 8. Conferir os checkpoints no Drive

In [ ]:
!ls -la "/content/drive/MyDrive/atelie_generativo/lora_config1_rank4"
!ls -la "/content/drive/MyDrive/atelie_generativo/lora_config2_rank8"

## Depois disso

- Confirmar os 2 modelos publicados no HF
- Preencher model card de cada um
- Guardar as imagens de validação pra grade comparativa da Etapa 3

```bash
git add notebooks/02_treino_lora.ipynb
git commit -m "notebook treino lora"
git push
```